In [0]:
# # Databricks notebook source
# import sys
# import os
 
# # We provide an empty default so the Job doesn't crash before our validation runs.
# dbutils.widgets.text("catalog_name", "")
# dbutils.widgets.text("raw_path", "")

# CATALOG = dbutils.widgets.get("catalog_name")
# RAW_PATH = dbutils.widgets.get("raw_path")

# print(f"Received CATALOG from Job: '{CATALOG}'")
# print(f"Received RAW_PATH from Job: '{RAW_PATH}'")

# # --- 1. Stable Package Import --- 
# workspace_root = os.path.abspath('../..')
# if workspace_root not in sys.path:
#     sys.path.append(workspace_root) 

# from src.config.paths import ProjectConfig

# # Initialize configuration. This will raise a ValueError if parameters are missing.
# cfg = ProjectConfig(CATALOG, RAW_PATH)
 
# # --- 2. Catalog & Schema Initialization --- 
# print(f"\nCreating -- CATALOG => {cfg.CATALOG}\n")
# spark.sql(f"CREATE CATALOG IF NOT EXISTS {cfg.CATALOG}")

# for schema in cfg.SCHEMAS: 
#     schema_name = f"{cfg.CATALOG}.{schema}"    
#     print(f"Creating -- SCHEMA => {schema_name}") 
#     spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")    

# # --- 3. Volume Initialization (The Raw & Checkpoint Zones) ---
# print(f"\nCreating -- VOLUME => {cfg.RAW_SCHEMA}.vol_raw")
# spark.sql(f"""
#     CREATE EXTERNAL VOLUME IF NOT EXISTS {cfg.RAW_SCHEMA}.vol_raw
#     LOCATION '{cfg.RAW_PATH}'
#     COMMENT 'Raw zone for raw files ingestion';
# """)          

# print(f"Creating -- VOLUME => {cfg.BRONZE_SCHEMA}.vol_bronze")
# spark.sql(f"CREATE VOLUME IF NOT EXISTS {cfg.BRONZE_SCHEMA}.vol_bronze")
 
# print(f"Creating -- VOLUME => {cfg.SILVER_SCHEMA}.vol_silver")
# spark.sql(f"CREATE VOLUME IF NOT EXISTS {cfg.SILVER_SCHEMA}.vol_silver")

# print(f"Creating -- VOLUME => {cfg.GOLD_SCHEMA}.vol_gold")
# spark.sql(f"CREATE VOLUME IF NOT EXISTS {cfg.GOLD_SCHEMA}.vol_gold")


# # --- 4. Table Initialization (Triggering Child Notebooks) ---
# # We use dbutils.notebook.run with relative paths (./) for DAB stability
# print("\nBuilding Bronze Layer Tables...")
# dbutils.notebook.run("./01_create_tables_bronze.py", 600, {"catalog_name": cfg.CATALOG})

# print("Building Silver Layer Tables...")
# dbutils.notebook.run("./02_create_tables_silver", 600, {"catalog_name": cfg.CATALOG})

# print("Building Gold Layer Tables...")
# dbutils.notebook.run("./03_create_tables_gold", 600, {"catalog_name": cfg.CATALOG})

# print("\n--- LENDING RISK ENVIRONMENT READY ---")

In [0]:
# Databricks notebook source
import sys
import os

# --- 1. Parameter Retrieval (Environment Layer) ---
# We provide an empty default so the Job doesn't crash before our validation runs.
dbutils.widgets.text("catalog_name", "")
dbutils.widgets.text("raw_path", "")

CATALOG = dbutils.widgets.get("catalog_name")
RAW_PATH = dbutils.widgets.get("raw_path")

print(f"Received CATALOG from Job: '{CATALOG}'")
print(f"Received RAW_PATH from Job: '{RAW_PATH}'")

# --- 2. Dynamic Root Discovery (Fixes the src import issue) --- 
# Instead of hardcoded '../..', we climb the tree until we find the project root.
project_root = os.getcwd()
while project_root != '/' and not os.path.exists(os.path.join(project_root, 'src')):
    project_root = os.path.dirname(project_root)

if project_root not in sys.path:
    # We use insert(0) to ensure your local src takes priority over pre-installed libs
    sys.path.insert(0, project_root) 

from src.config.paths import ProjectConfig

# Initialize configuration. This will raise a ValueError if parameters are missing.
cfg = ProjectConfig(CATALOG, RAW_PATH)
 
# --- 3. Catalog & Schema Initialization --- 
print(f"\nCreating -- CATALOG => {cfg.CATALOG}\n")
spark.sql(f"CREATE CATALOG IF NOT EXISTS {cfg.CATALOG}")

for schema in cfg.SCHEMAS: 
    schema_name = f"{cfg.CATALOG}.{schema}"    
    print(f"Creating -- SCHEMA => {schema_name}") 
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")    

# --- 4. Volume Initialization (The Raw & Checkpoint Zones) ---
print(f"\nCreating -- VOLUME => {cfg.RAW_SCHEMA}.vol_raw")
spark.sql(f"""
    CREATE EXTERNAL VOLUME IF NOT EXISTS {cfg.RAW_SCHEMA}.vol_raw
    LOCATION '{cfg.RAW_PATH}'
    COMMENT 'Raw zone for raw files ingestion';
""")          

print(f"Creating -- VOLUME => {cfg.BRONZE_SCHEMA}.vol_bronze")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {cfg.BRONZE_SCHEMA}.vol_bronze")
 
print(f"Creating -- VOLUME => {cfg.SILVER_SCHEMA}.vol_silver")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {cfg.SILVER_SCHEMA}.vol_silver")

print(f"Creating -- VOLUME => {cfg.GOLD_SCHEMA}.vol_gold")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {cfg.GOLD_SCHEMA}.vol_gold")

# --- 5. Table Initialization (Triggering Child Notebooks) ---
# Note: Removed '.py' extensions for dbutils.notebook.run as they are redundant in DABs
# and added raw_path to params to satisfy ProjectConfig in child notebooks.
child_params = {"catalog_name": cfg.CATALOG, "raw_path": cfg.RAW_PATH}

print("\nBuilding Bronze Layer Tables...")
dbutils.notebook.run("./01_create_tables_bronze", 600, child_params)

print("Building Silver Layer Tables...")
dbutils.notebook.run("./02_create_tables_silver", 600, child_params)

print("Building Gold Layer Tables...")
dbutils.notebook.run("./03_create_tables_gold", 600, child_params)

print("\n--- LENDING RISK ENVIRONMENT READY ---")